# 03 Null test
Does a method flag a pool that hides nothing? For each of 101 real pools (12 atlases) we built a twin by deleting every hidden cell. A method's evidence for a pool is the mean score of its 200 top cells (for our method, which blends two ranks and has no native scale, the learned scorer's score on the blend's top 200). The threshold flags 95% of the real pools; the false-call rate is the share of twins above it. An uninformative score gives about 0.95, lower is better.

Inputs: `predictions.tgz` (scores of every method on real and twin pools, and `ids/twin_manifest.csv`). Rates are atlas-equal (every atlas counts the same) with the pool-weighted rate alongside.

In [1]:
import os, sys, warnings

warnings.filterwarnings("ignore")
REPO = os.path.abspath(os.environ.get("REPO", ".."))
ASSETS = os.path.abspath(os.environ.get("ASSETS", os.path.join(REPO, "assets")))
sys.path.insert(0, REPO)
sys.dont_write_bytecode = True
import json, numpy as np, pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
EXP = f"{REPO}/eval/expected"
from eval import nulltest as N

tw = pd.read_csv(f"{ASSETS}/ids/twin_manifest.csv")
assert len(tw) == 101 and tw.atlas.nunique() == 12
methods = [
    "hiddenrare",
    "scorer",
    "leiden_sub",
    "iforest_pca",
    "lof_pca",
    "iforest_scvi",
    "uncertainty",
    "dup_loop",
    "nodup_loop",
    "classweight_loop",
    "seprank",
]
R = pd.DataFrame(N.run(f"{ASSETS}/predictions", methods, tw)).T
print(R.sort_values("atlas_equal").round(3))

                  atlas_equal  pool_weighted
iforest_scvi            0.806          0.772
iforest_pca             0.868          0.921
lof_pca                 0.884          0.891
dup_loop                0.892          0.911
leiden_sub              0.896          0.901
seprank                 0.908          0.861
uncertainty             0.911          0.921
nodup_loop              0.918          0.931
hiddenrare              0.930          0.931
classweight_loop        0.952          0.950
scorer                  0.972          0.970


## 1. The 101 pools of 12 atlases
Compared with the stored values of the analysis (`eval/expected/null_false_calls.json`) to 1e-9, and with the three rates the paper quotes.

In [2]:
exp = pd.DataFrame(json.load(open(f"{EXP}/null_false_calls.json"))).T.loc[R.index]
assert np.allclose(R.values.astype(float), exp[R.columns].values, atol=1e-9, rtol=0)
r = R.atlas_equal.round(3)
assert (r["hiddenrare"], r["leiden_sub"], r["iforest_scvi"]) == (
    0.930,
    0.896,
    0.806,
) and r.idxmin() == "iforest_scvi"
print(
    "false calls: ours %.3f, Leiden subclustering %.3f, isolation forest on scVI %.3f (lowest), 0.95 for an uninformative score"
    % (r["hiddenrare"], r["leiden_sub"], r["iforest_scvi"])
)

false calls: ours 0.930, Leiden subclustering 0.896, isolation forest on scVI 0.806 (lowest), 0.95 for an uninformative score


## 2. The 69 pools of the 8-atlas test set
The same readout on the twins whose real pool is one of the 69 held-out pools, a post hoc recomputation of the paper, checked against `eval/expected/null_false_calls_69.json` to 1e-9. Then the area under the curve (AUC) for real against twin pools, from each pool's evidence and pooled over the 101 pairs: the paper prints 0.60 for both methods and the pooled AUC recomputes to 0.61.

In [3]:
from eval import paper as P
from sklearn.metrics import roc_auc_score

pids, _ = P.pools(ASSETS)
t69 = tw[tw.real_pool.isin(pids)]
assert len(t69) == 69
R69 = pd.DataFrame(
    N.run(f"{ASSETS}/predictions", [m for m in methods if m != "scorer"], t69)
).T
e69 = pd.DataFrame(json.load(open(f"{EXP}/null_false_calls_69.json"))).T.loc[R69.index]
assert np.allclose(R69.values.astype(float), e69[R69.columns].values, atol=1e-9, rtol=0)
r = R69.atlas_equal.round(3)
assert (r["hiddenrare"], r["leiden_sub"]) == (0.822, 0.864)
print(R69.sort_values("atlas_equal").round(3))
auc = {}
for m in ("hiddenrare", "leiden_sub"):
    re = [N.evidence(f"{ASSETS}/predictions", m, p) for p in tw.real_pool]
    te = [N.evidence(f"{ASSETS}/predictions", m, p) for p in tw.pilot_id]
    auc[m] = float(roc_auc_score([1] * len(re) + [0] * len(te), re + te))
assert (round(auc["hiddenrare"], 3), round(auc["leiden_sub"], 3)) == (0.612, 0.610)
print("real-vs-twin AUC (pooled):", {k: round(v, 3) for k, v in auc.items()})

                  atlas_equal  pool_weighted
iforest_pca             0.792          0.913
hiddenrare              0.822          0.826
iforest_scvi            0.828          0.739
seprank                 0.832          0.812
leiden_sub              0.864          0.884
uncertainty             0.885          0.913
dup_loop                0.888          0.913
lof_pca                 0.903          0.884
nodup_loop              0.928          0.928
classweight_loop        0.944          0.942


real-vs-twin AUC (pooled): {'hiddenrare': np.float64(0.612), 'leiden_sub': np.float64(0.61)}
